# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameyachopade/1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from google.colab import userdata

# Ensure we have the required libraries
!pip install -q datasets huggingface_hub pyarrow pandas

# Retrieve Hugging Face token securely from secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face!")
except Exception as e:
    print(f"Could not authenticate: {e}. Please make sure you saved your token as 'HF_TOKEN' in the Secrets tab (🔑).")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!


In [2]:
from datasets import load_dataset
import pandas as pd

# Let's inspect the files available in the dataset
from huggingface_hub import list_repo_files
try:
    files = list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")
    print("Files found in the warehouse repository:")
    for f in files[:15]:
        print(f" - {f}")
except Exception as e:
    print(f"Error fetching dataset repository details: {e}")

Files found in the warehouse repository:
 - .gitattributes
 - README.md
 - dim_clients.parquet
 - dim_content.parquet
 - fact_content_daily_performance/month=2025-01/data_0.parquet
 - fact_content_daily_performance/month=2025-02/data_0.parquet
 - fact_content_daily_performance/month=2025-03/data_0.parquet
 - fact_content_daily_performance/month=2025-04/data_0.parquet
 - fact_content_daily_performance/month=2025-05/data_0.parquet
 - fact_content_daily_performance/month=2025-06/data_0.parquet
 - fact_content_daily_performance/month=2025-07/data_0.parquet
 - fact_content_daily_performance/month=2025-08/data_0.parquet
 - fact_content_daily_performance/month=2025-09/data_0.parquet
 - fact_content_daily_performance/month=2025-10/data_0.parquet
 - fact_content_daily_performance/month=2025-11/data_0.parquet


In [3]:
from huggingface_hub import list_repo_files
all_files = list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")
perf_files = [f for f in all_files if "fact_content_daily_performance" in f]
print(f"Total performance files: {len(perf_files)}")
print("First 5 performance files:", perf_files[:5])
print("Last 5 performance files:", perf_files[-5:])

Total performance files: 19
First 5 performance files: ['fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet']
Last 5 performance files: ['fact_content_daily_performance/month=2026-03/data_0.parquet', 'fact_content_daily_performance/month=2026-04/data_0.parquet', 'fact_content_daily_performance/month=2026-05/data_0.parquet', 'fact_content_daily_performance/month=2026-06/data_0.parquet', 'fact_content_daily_performance_sample.parquet']


In [4]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
import pandas as pd

try:
    dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename="dim_content.parquet")
    dim_clients_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename="dim_clients.parquet")
    dim_content = pd.read_parquet(dim_content_path)
    dim_clients = pd.read_parquet(dim_clients_path)
    print("dim_content columns:", dim_content.columns.tolist())
    print("dim_clients columns:", dim_clients.columns.tolist())

    # Pick a sample month like 2026-03
    target_file = "fact_content_daily_performance/month=2026-03/data_0.parquet"
    if target_file in all_files:
        fact_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename=target_file)
        fact_df = pd.read_parquet(fact_path)
        print("fact_df shape (month=2026-03):", fact_df.shape)
        print("fact_df columns:", fact_df.columns.tolist())
    else:
        # Fallback to a month that exists in the list
        fallback_file = [f for f in perf_files if "2025" in f or "2026" in f][0]
        print(f"Using fallback file: {fallback_file}")
        fact_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename=fallback_file)
        fact_df = pd.read_parquet(fact_path)
        print("fact_df shape:", fact_df.shape)
        print("fact_df columns:", fact_df.columns.tolist())
except Exception as e:
    print("Error loading tables:", e)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
dim_clients columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_df shape (month=2026-03): (9841378, 30)
fact_df columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [5]:
import pandas as pd
from datasets import load_dataset

# Let's read a sample month of daily performance data to see the schema and content
# We'll use month=2026-03 (or similar mid-panel month available in the repo)
try:
    # Find all months available
    all_files = list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")
    perf_files = [f for f in all_files if "fact_content_daily_performance" in f]
    print(f"Total monthly files: {len(perf_files)}")
    print("Sample performance files:")
    for f in perf_files[:5]:
        print(f" - {f}")
    for f in perf_files[-5:]:
        print(f" - {f}")
except Exception as e:
    print(f"Error: {e}")

Total monthly files: 19
Sample performance files:
 - fact_content_daily_performance/month=2025-01/data_0.parquet
 - fact_content_daily_performance/month=2025-02/data_0.parquet
 - fact_content_daily_performance/month=2025-03/data_0.parquet
 - fact_content_daily_performance/month=2025-04/data_0.parquet
 - fact_content_daily_performance/month=2025-05/data_0.parquet
 - fact_content_daily_performance/month=2026-03/data_0.parquet
 - fact_content_daily_performance/month=2026-04/data_0.parquet
 - fact_content_daily_performance/month=2026-05/data_0.parquet
 - fact_content_daily_performance/month=2026-06/data_0.parquet
 - fact_content_daily_performance_sample.parquet


In [6]:
# Let's load the dim tables and one mid-panel performance month (e.g. 2026-03 if exists, or latest available from 2026)
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

try:
    # Download dim tables
    dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename="dim_content.parquet")
    dim_clients_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename="dim_clients.parquet")

    dim_content = pd.read_parquet(dim_content_path)
    dim_clients = pd.read_parquet(dim_clients_path)

    print("dim_content shape:", dim_content.shape)
    print("dim_content columns:", list(dim_content.columns))
    print("dim_clients shape:", dim_clients.shape)
    print("dim_clients columns:", list(dim_clients.columns))

    # Let's try downloading fact data for March 2026 (month=2026-03)
    target_file = "fact_content_daily_performance/month=2026-03/data_0.parquet"
    if target_file in all_files:
        fact_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename=target_file)
        fact_df = pd.read_parquet(fact_path)
        print("fact_df shape (month=2026-03):", fact_df.shape)
        print("fact_df columns:", list(fact_df.columns))
    else:
        print("Month 2026-03 file not found, let's find a valid month.")
except Exception as e:
    print(f"Error reading parquet files: {e}")

dim_content shape: (519606, 26)
dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
dim_clients shape: (104, 9)
dim_clients columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
fact_df shape (month=2026-03): (9841378, 30)
fact_df columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_po

In [7]:
import glob
print("Notebook files found in workspace:")
print(glob.glob('**/*.ipynb', recursive=True))

Notebook files found in workspace:
[]


## 1. Unit of analysis + time window

### The Contract Answers:
1. **What one row means for your lane (the grain):** One row represents a single piece of content (uniquely identified by `content_hash_id`) for a specific client (`client_hash_id`) observed on a single day (`report_date`).
2. **Which table(s) you'll use:** `fact_content_daily_performance` joined with dimension tables `dim_content` and `dim_clients` using `content_hash_id` and `client_hash_id` respectively.
3. **Which time window:** A mid-panel month of **March 2026** (`month=2026-03`).
4. **What you would predict or rank:** Organic sessions (`sessions_organic`) or GSC clicks (`gsc_clicks`) as proxies for traffic, or ranking position (`gsc_avg_position`).
5. **What you deliberately exclude:** We exclude rows where the client does not have active integration or when `is_deleted` is TRUE in the content dimension.

In [8]:
# Let's programmatically verify that the combination of (report_date, client_hash_id, content_hash_id) is unique (the grain)
is_unique = fact_df.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size().max() == 1
print(f"Is the (report_date, client_hash_id, content_hash_id) combination unique? {is_unique}")
print(f"Total unique rows in 2026-03: {len(fact_df):,}")

Is the (report_date, client_hash_id, content_hash_id) combination unique? True
Total unique rows in 2026-03: 9,841,378


## 2. Fields: feature / label / context / excluded

### Field Buckets:
- **Features (Knowable at the decision moment):**
  - `keyword_char_count` (length of target query)
  - `search_volume` (organic opportunity scale)
  - `competition_level` / `competition` (ad bidding difficulty proxy)
  - `cpc` (commercial intent value)
  - `backlinks` (content authority score)
- **Label:** `sessions_organic` (actual measured search visits for that day)
- **Context:** `report_date`, `client_hash_id`, `content_hash_id`, `main_intent`
- **Excluded:**
  - `scroll_events`, `ga4_pageviews`, `ga4_sessions`, `gsc_clicks` (deliberately excluded during feature engineering to avoid target leakage as they happen simultaneously with or downstream from the target event).

In [9]:
# Show column counts and type matches for verifying our schema fields
print(fact_df[['report_date', 'sessions_organic', 'client_has_gsc', 'client_has_ga4']].head())

  report_date  sessions_organic  client_has_gsc  client_has_ga4
0  2026-03-01               NaN            True           False
1  2026-03-01               NaN            True           False
2  2026-03-01               NaN            True           False
3  2026-03-01               NaN            True           False
4  2026-03-01               NaN            True           False


## 3. Verify it with queries (grain, counts, missing values, windows)

### Let's verify our three key facts with real operations:
1. **Grain verification:** Verify there are no duplicate entries for the primary keys.
2. **Row count & date span:** Validate row count and date boundaries for March 2026.
3. **Availability & filter:** Check availability flags (`client_has_gsc IS TRUE` & `client_has_ga4 IS TRUE`) and count the survival rate.

In [10]:
# Fact 1: Grain verification
dups = fact_df.duplicated(subset=['report_date', 'client_hash_id', 'content_hash_id']).sum()
print(f"Fact 1 (Grain): Duplicates found = {dups}")

# Fact 2: Row count & Date span
min_date = fact_df['report_date'].min()
max_date = fact_df['report_date'].max()
print(f"Fact 2 (Date Span): Start = {min_date}, End = {max_date}, Rows = {len(fact_df):,}")

# Fact 3: Integration Availability Survival
surviving_df = fact_df[(fact_df['client_has_gsc'] == True) & (fact_df['client_has_ga4'] == True)]
surviving_pct = len(surviving_df) / len(fact_df) * 100
print(f"Fact 3 (Availability): Rows with active GSC & GA4 = {len(surviving_df):,} ({surviving_pct:.2f}% survival rate)")

Fact 1 (Grain): Duplicates found = 0
Fact 2 (Date Span): Start = 2026-03-01, End = 2026-03-31, Rows = 9,841,378
Fact 3 (Availability): Rows with active GSC & GA4 = 6,822,637 (69.33% survival rate)


## 4. Five Features + Leakage Trap Experiment

We build a clean five-feature frame and run our leakage experiment to observe how leaking future label indicators artifically inflates our validation predictive scores, then we clean it up.

In [2]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Ensure data is loaded if variables are not in scope
if 'fact_df' not in locals() or 'dim_content' not in locals():
    print("Loading dataset files programmatically...")
    try:
        dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename="dim_content.parquet")
        dim_content = pd.read_parquet(dim_content_path)
        target_file = "fact_content_daily_performance/month=2026-03/data_0.parquet"
        fact_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", filename=target_file)
        fact_df = pd.read_parquet(fact_path)
    except Exception as e:
        print(f"Error loading datasets: {e}")

print("Step 1: Merging fact data and dimensions...")
merged = fact_df.merge(dim_content, on=['client_hash_id', 'content_hash_id'], how='inner')

# Select 5 features (all knowable prior to or at the decision moment)
feature_cols = ['keyword_char_count', 'search_volume', 'cpc', 'backlinks', 'word_count']

# Fill missing values to avoid modeling failures
for col in feature_cols:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors='coerce').fillna(0)

# Define target
merged['target'] = (pd.to_numeric(merged['sessions_organic'], errors='coerce').fillna(0) > 0).astype(int)

# Step 2: The Leakage Trap
# Introduce 'ga4_pageviews' which is a concurrent activity metric directly linked to organic sessions!
merged['leakage_feature'] = pd.to_numeric(merged['ga4_pageviews'], errors='coerce').fillna(0)

print("Step 2: Subsampling data to run modeling rapidly...")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sample_subset = merged.sample(n=20000, random_state=42)

# Model 1: With Leakage
X_leak = sample_subset[feature_cols + ['leakage_feature']]
y = sample_subset['target']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42)
clf_leak = RandomForestClassifier(max_depth=5, n_estimators=30, random_state=42).fit(X_train_l, y_train_l)
score_leak = roc_auc_score(y_test_l, clf_leak.predict_proba(X_test_l)[:, 1])
print(f"Model WITH leakage ROC-AUC score: {score_leak:.4f}")

# Model 2: Clean and Honest
X_clean = sample_subset[feature_cols]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y, test_size=0.3, random_state=42)
clf_clean = RandomForestClassifier(max_depth=5, n_estimators=30, random_state=42).fit(X_train_c, y_train_c)
score_clean = roc_auc_score(y_test_c, clf_clean.predict_proba(X_test_c)[:, 1])
print(f"Model WITHOUT leakage (Honest) ROC-AUC score: {score_clean:.4f}")

Loading dataset files programmatically...
Step 1: Merging fact data and dimensions...
Step 2: Subsampling data to run modeling rapidly...
Model WITH leakage ROC-AUC score: 0.9937
Model WITHOUT leakage (Honest) ROC-AUC score: 0.7869


## 5. Limitations & Self-Check

### Limitation of the Slice:
Our current slice relies heavily on external static search metrics (`search_volume`, `cpc`) and dimensions which do not change day-to-day. Since daily variations are driven by ranking fluctuations and seasonality, static features struggle to explain precise daily variations unless historical rolling-window lags are computed.

### Self-Check Status:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are shown
- [x] Claims use precise, objective terms
- [x] Committed to my repo under `work/notebooks/w03_data_contract.ipynb`